# 01 — Data Preparation

Load, validate, and clean all raw data files. Outputs processed CSVs to `data/processed/`.

**Files loaded:**
- `BasicPay_2026.csv` — 2026 DFAS basic pay table
- `PromotionTiming.csv` — grade at each YOS for all three career profiles
- `ActiveDutyWithdrawalRates.csv` — separation and survival rates by YOS
- `TSP_FundPerformance.csv` — annual TSP fund returns (metadata rows stripped, % → float)
- `CPI.csv` — annual CPI index values; year-over-year inflation rates derived
- `LifeExpectancy_2022.csv` — SSA 2022 actuarial life tables

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(exist_ok=True)

## 1. Basic Pay Table (`BasicPay_2026.csv`)

In [2]:
basic_pay = pd.read_csv(RAW / "BasicPay_2026.csv", index_col="PayGrade")

# Rename YOS columns from strings to ints for easier indexing downstream
basic_pay.columns = basic_pay.columns.astype(int)

null_count = basic_pay.isna().sum().sum()
print(f"Shape: {basic_pay.shape}  (pay grades × YOS breakpoints)")
print(f"Pay grades: {basic_pay.index.tolist()}")
print(f"YOS breakpoints: {basic_pay.columns.tolist()}")
print(f"\nNull cells (grade/YOS combos that don't exist): {null_count}")
print(f"\nSpot-check — O-3 at YOS 6: ${basic_pay.loc['O-3', 6]:,.2f}")
print(f"Spot-check — E-5 at YOS 8: ${basic_pay.loc['E-5', 8]:,.2f}")
basic_pay

Shape: (22, 22)  (pay grades × YOS breakpoints)
Pay grades: ['O-10', 'O-9', 'O-8', 'O-7', 'O-6', 'O-5', 'O-4', 'O-3', 'O-2', 'O-1', 'O-3E', 'O-2E', 'O-1E', 'E-9', 'E-8', 'E-7', 'E-6', 'E-5', 'E-4', 'E-3', 'E-2', 'E-1']
YOS breakpoints: [0, 2, 3, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40]

Null cells (grade/YOS combos that don't exist): 42

Spot-check — O-3 at YOS 6: $7,737.00
Spot-check — E-5 at YOS 8: $4,299.90


,0,2,3,4,6,8,10,12,14,16,...,22,24,26,28,30,32,34,36,38,40
PayGrade,,,,,,,,,,,,,,,,,,,,,
O-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,18999.9,18999.9,18999.9,18999.9,18999.9,18999.9,18999.9,18999.9,18999.9,18999.9
O-9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,18999.9,18999.9,18999.9,18999.9,18999.9,18999.9,18999.9,18999.9,18999.9,18999.9
O-8,13888.5,14343.9,14645.4,14729.4,15106.5,15735.3,15882.0,16479.6,16651.8,17166.6,...,18999.9,18999.9,18999.9,18999.9,18999.9,18999.9,18999.9,18999.9,18999.9,18999.9
O-7,11540.1,12076.2,12324.3,12522.0,12878.7,13231.8,13639.2,14045.7,14454.3,15735.3,...,16817.7,16817.7,16904.4,16904.4,17242.2,17242.2,17242.2,17242.2,17242.2,17242.2
O-6,8751.3,9613.8,10245.0,10245.0,10284.3,10725.0,10783.5,10783.5,11396.4,12479.7,...,14112.9,14479.2,15188.7,15188.7,15408.3,15408.3,15408.3,15408.3,15408.3,15408.3
O-5,7295.4,8218.2,8787.0,8894.1,9249.6,9461.4,9928.5,10271.7,10715.1,11391.3,...,12394.8,12394.8,12394.8,12394.8,12394.8,12394.8,12394.8,12394.8,12394.8,12394.8
O-4,6294.6,7286.4,7773.6,7881.0,8332.2,8816.4,9420.0,9888.3,10214.4,10401.6,...,10509.9,10509.9,10509.9,10509.9,10509.9,10509.9,10509.9,10509.9,10509.9,10509.9
O-3,5534.1,6273.9,6770.4,7382.7,7737.0,8125.5,8375.7,8788.2,9004.2,9004.2,...,9004.2,9004.2,9004.2,9004.2,9004.2,9004.2,9004.2,9004.2,9004.2,9004.2
O-2,4782.0,5446.2,6272.4,6484.5,6617.7,6617.7,6617.7,6617.7,6617.7,6617.7,...,6617.7,6617.7,6617.7,6617.7,6617.7,6617.7,6617.7,6617.7,6617.7,6617.7


## 2. Promotion Timing (`PromotionTiming.csv`)

In [3]:
promotion = pd.read_csv(RAW / "PromotionTiming.csv")

print(f"Shape: {promotion.shape}")
print(f"YOS range: {promotion['YOS'].min()}–{promotion['YOS'].max()}")

# dropna() required: Enlisted is blank for YOS 31–40 (max service = 30 years)
officer_grades = sorted(promotion['Officer'].dropna().unique())
enlisted_grades = sorted(promotion['Enlisted'].dropna().unique())
peo_grades = sorted(promotion['PriorEnlistedOfficer'].dropna().unique())
print(f"\nUnique officer grades:  {officer_grades}")
print(f"Unique enlisted grades: {enlisted_grades}")
print(f"Unique PEO grades:      {peo_grades}")
print(f"\nNull rows per column:\n{promotion.isna().sum().to_string()}")

# Confirm PEO profile transitions from enlisted to officer at YOS 9
peo_transition = promotion.loc[
    promotion['PriorEnlistedOfficer'].str.startswith('O', na=False),
    'YOS'
].min()
print(f"\nPEO transitions to officer grade at YOS: {peo_transition}")
promotion.style.hide(axis="index").format(na_rep="")

Shape: (40, 4)
YOS range: 1–40

Unique officer grades:  ['O-1', 'O-2', 'O-3', 'O-4', 'O-5', 'O-6', 'O-7', 'O-8']
Unique enlisted grades: ['E-2', 'E-3', 'E-4', 'E-5', 'E-6', 'E-7', 'E-8', 'E-9']
Unique PEO grades:      ['E-2', 'E-3', 'E-4', 'E-5', 'O-1E', 'O-2E', 'O-3E', 'O-4', 'O-5', 'O-6', 'O-7', 'O-8']

Null rows per column:
YOS                      0
Officer                  0
Enlisted                10
PriorEnlistedOfficer     0

PEO transitions to officer grade at YOS: 9


YOS,Officer,Enlisted,PriorEnlistedOfficer
1,O-1,E-2,E-2
2,O-2,E-3,E-3
3,O-2,E-3,E-3
4,O-3,E-4,E-4
5,O-3,E-4,E-4
6,O-3,E-5,E-5
7,O-3,E-5,E-5
8,O-3,E-5,E-5
9,O-3,E-6,O-1E
10,O-4,E-6,O-2E


## 3. Active Duty Withdrawal Rates (`ActiveDutyWithdrawalRates.csv`)

In [4]:
withdrawal = pd.read_csv(RAW / "ActiveDutyWithdrawalRates.csv")

print(f"Shape: {withdrawal.shape}")
print(f"YOS range: {withdrawal['YOS'].min()}–{withdrawal['YOS'].max()}")

# Survival at YOS N is calculated AFTER applying that year's withdrawal rate:
#   S[N] = S[N-1] * (1 - W[N])
# S[19] = fraction who enter year 20, i.e., "reach 20 years" per DoD.
# S[20] = fraction continuing past 20 years — much lower due to the
# large YOS-20 retirement spike.
row_19 = withdrawal.loc[withdrawal['YOS'] == 19].squeeze()
row_20 = withdrawal.loc[withdrawal['YOS'] == 20].squeeze()

print(f"\nSurvival entering YOS 20 (end of YOS 19) — 'reach 20 years':")
print(f"  Officer:  {row_19['OfficerSurvival']:.1%}  (expected ~41%)")
print(f"  Enlisted: {row_19['EnlistedSurvival']:.1%}  (expected ~19%)")

print("\nSurvival past YOS 20 (beyond 20 years, after retirement spike):")
print(f"  Officer:  {row_20['OfficerSurvival']:.1%}")
print(f"  Enlisted: {row_20['EnlistedSurvival']:.1%}")

Shape: (40, 5)
YOS range: 1–40

Survival entering YOS 20 (end of YOS 19) — 'reach 20 years':
  Officer:  41.2%  (expected ~41%)
  Enlisted: 18.8%  (expected ~19%)

Survival past YOS 20 (beyond 20 years, after retirement spike):
  Officer:  29.1%
  Enlisted: 10.4%


## 4. TSP Fund Performance (`TSP_FundPerformance.csv`)

The first six data rows are summary metadata (inception date, 1yr, 3yr, 5yr, 10yr, since-inception averages).
These are skipped; only rows with a numeric 4-digit year are retained.
Percentage strings are stripped and converted to float.

In [5]:
tsp_raw = pd.read_csv(RAW / "TSP_FundPerformance.csv")

# Keep only rows where Year is a 4-digit integer
year_mask = pd.to_numeric(tsp_raw["Year"], errors="coerce").notna()
tsp = tsp_raw[year_mask].copy()
tsp["Year"] = tsp["Year"].astype(int)

# Convert percentage strings to floats ("nan" strings coerce to NaN)
pct_cols = [c for c in tsp.columns if c != "Year"]
for col in pct_cols:
    cleaned = (
        tsp[col]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.strip()
    )
    tsp[col] = pd.to_numeric(cleaned, errors="coerce")

tsp = tsp.sort_values("Year").reset_index(drop=True)

print(f"Year range: {tsp['Year'].min()}–{tsp['Year'].max()}")
print(f"Rows (annual observations): {len(tsp)}")
print(f"Metadata rows dropped: {year_mask.sum() != tsp_raw.shape[0]}  "
      f"({tsp_raw.shape[0] - year_mask.sum()} rows skipped)")
tsp.tail()

Year range: 1987–2025
Rows (annual observations): 39
Metadata rows dropped: True  (6 rows skipped)


,Year,L Income,L 2030,L 2035,L 2040,L 2045,L 2050,L 2055,L 2060,L 2065,...,L 2075,G Fund,F Fund,U.S. Aggregate Index,C Fund,S&P 500,S Fund,DJ TSM,I Fund,International Index
34,2021,5.42,12.37,13.43,14.51,15.40,16.34,19.90,19.90,19.90,...,NaN,1.38,-1.46,-1.54,28.68,28.71,12.45,12.35,11.45,11.26
35,2022,-2.70,-10.32,-11.65,-12.90,-14.03,-15.05,-17.60,-17.61,-17.62,...,NaN,2.98,-12.83,-13.01,-18.13,-18.11,-26.26,-26.54,-13.94,-14.45
36,2023,8.99,15.76,16.91,18.04,19.03,20.00,23.31,23.30,23.31,...,NaN,4.22,5.58,5.53,26.25,26.29,25.30,24.97,18.38,18.24
37,2024,7.37,11.52,12.18,12.85,13.42,14.02,16.28,16.28,16.28,...,NaN,4.40,1.33,1.25,24.96,25.02,16.93,16.88,4.27,4.20
38,2025,9.36,15.17,16.27,17.31,18.20,19.07,21.87,21.88,21.88,...,NaN,4.44,7.21,7.30,17.85,17.88,11.38,11.32,32.45,32.01


In [6]:
# L Fund data coverage — determines which funds to use in the glide path model
l_funds = [c for c in tsp.columns if c.startswith("L")]

print("L Fund annual return data availability (minimum 10 years recommended):")
header = (
    f"{'Fund':<14} {'Non-null years':>15}  "
    f"{'First year':>12}  {'Sufficient (>=10yr)':>20}"
)
print(header)
print("-" * 65)
for fund in l_funds:
    n = tsp[fund].notna().sum()
    first_yr = (
        tsp.loc[tsp[fund].notna(), "Year"].min() if n > 0 else "N/A"
    )
    flag = "YES" if n >= 10 else "NO — insufficient"
    print(f"  {fund:<12} {n:>15}  {str(first_yr):>12}  {flag:>20}")

L Fund annual return data availability (minimum 10 years recommended):
Fund            Non-null years    First year   Sufficient (>=10yr)
-----------------------------------------------------------------
  L Income                  20          2006                   YES
  L 2030                    20          2006                   YES
  L 2035                     5          2021     NO — insufficient
  L 2040                    20          2006                   YES
  L 2045                     5          2021     NO — insufficient
  L 2050                    14          2012                   YES
  L 2055                     5          2021     NO — insufficient
  L 2060                     5          2021     NO — insufficient
  L 2065                     5          2021     NO — insufficient
  L 2070                     1          2025     NO — insufficient
  L 2075                     0           N/A     NO — insufficient


## 4b. Synthetic L Fund Backfill (2002+)

L Income, L 2030, L 2040, and L 2050 are weighted
combinations of the five underlying individual funds.
Constrained least squares recovers those weights
(R² 0.989–0.998), which are then used to compute synthetic
returns for years where individual fund data exists but the
L Fund does not.

- S Fund and I Fund data starts 2002, so the backfill
  extends all four L Funds back to 2002 (n: 14-20 → 24).
- Actual L Fund values (where they exist) are unchanged.
- L 2050 is most affected: its 14-year window missed the
  2001–2002 tech bust, biasing its std downward.

In [7]:
from scipy.optimize import minimize

IND = ['G Fund', 'F Fund', 'C Fund', 'S Fund', 'I Fund']
L_TARGET = ['L Income', 'L 2030', 'L 2040', 'L 2050']

# Capture original stats before backfill
orig = {
    lf: {
        'n':    int(tsp[lf].notna().sum()),
        'mean': float(tsp[lf].dropna().mean()),
        'std':  float(tsp[lf].dropna().std(ddof=1)),
    }
    for lf in L_TARGET
}

# Fit weights from overlapping years: constrained least
# squares (weights >= 0, sum to 1) so the coefficients are
# interpretable as portfolio allocations
synth_weights = {}
r_squared = {}
for lf in L_TARGET:
    overlap = tsp[IND + [lf]].dropna()
    X, y = overlap[IND].values, overlap[lf].values
    def obj(w):
        return np.sum((y - X @ w) ** 2)
    res = minimize(
        obj, np.ones(5) / 5, method='SLSQP',
        bounds=[(0, 1)] * 5,
        constraints=[{'type': 'eq',
                       'fun': lambda w: w.sum() - 1}],
    )
    synth_weights[lf] = res.x
    # res.fun is the residual sum of squares at the optimum
    ss_tot = np.sum((y - y.mean()) ** 2)
    r_squared[lf] = 1 - res.fun / ss_tot

# Display implied allocations
col = ('{:<12} {:>7} {:>7} {:>7} {:>7} {:>7}'
       '  {:>7} {:>7} {:>8}')
print('Implied allocations (constrained least squares):')
print(col.format('L Fund', 'G', 'F', 'C', 'S', 'I',
                 'Stocks', 'Bonds', 'R^2'))
print('-' * 81)
for lf in L_TARGET:
    g, f, c, s, i = synth_weights[lf]
    print(col.format(
        lf,
        f'{g:.1%}', f'{f:.1%}', f'{c:.1%}',
        f'{s:.1%}', f'{i:.1%}',
        f'{c+s+i:.1%}', f'{g+f:.1%}',
        f'{r_squared[lf]:.4f}',
    ))

# Backfill: synthetic values for years where all 5 funds
# have data but the L Fund does not
for lf in L_TARGET:
    mask = (
        tsp[IND].notna().all(axis=1) & tsp[lf].isna()
    )
    tsp.loc[mask, lf] = (
        tsp.loc[mask, IND].values @ synth_weights[lf]
    )

# Before/after comparison
col2 = '{:<12} {:>5} {:>8} {:>8}   {:>5} {:>8} {:>8}'
print('\nCoverage and distribution: original -> extended:')
print('             -- Original --         -- Extended --')
print(col2.format('L Fund', 'n', 'Mean', 'Std',
                  'n', 'Mean', 'Std'))
print('-' * 60)
for lf in L_TARGET:
    o_n = orig[lf]['n']
    o_mu = orig[lf]['mean']
    o_sg = orig[lf]['std']
    n_new = int(tsp[lf].notna().sum())
    m_new = tsp[lf].dropna().mean()
    s_new = tsp[lf].dropna().std(ddof=1)
    print(col2.format(
        lf,
        o_n, f'{o_mu:.2f}%', f'{o_sg:.2f}%',
        n_new, f'{m_new:.2f}%', f'{s_new:.2f}%',
    ))
print()
print(
    'Synthetic rows use actual individual-fund returns;'
    ' no data fabricated.'
)

Implied allocations (constrained least squares):
L Fund             G       F       C       S       I   Stocks   Bonds      R^2
---------------------------------------------------------------------------------
L Income       69.4%    9.9%   13.3%    1.2%    6.3%    20.8%   79.2%   0.9889
L 2030         33.4%    0.0%   26.3%   16.4%   23.9%    66.6%   33.4%   0.9904
L 2040         22.6%    0.0%   32.0%   18.3%   27.1%    77.4%   22.6%   0.9946
L 2050         15.5%    0.6%   38.2%   17.7%   28.0%    83.9%   16.1%   0.9982

Coverage and distribution: original -> extended:
             -- Original --         -- Extended --
L Fund           n     Mean      Std       n     Mean      Std
------------------------------------------------------------
L Income        20    4.68%    3.78%      24    4.80%    3.74%
L 2030          20    8.01%   11.68%      24    8.09%   11.89%
L 2040          20    8.92%   13.53%      24    8.93%   13.79%
L 2050          14   11.63%   11.66%      24    9.47%   14.8

## 5. CPI / Inflation (`CPI.csv`)

In [8]:
cpi = pd.read_csv(RAW / "CPI.csv")
cpi = cpi.sort_values("Year").reset_index(drop=True)

# Year-over-year inflation rate (percentage points)
cpi["Inflation"] = cpi["Annual"].pct_change() * 100

print(f"CPI year range: {cpi['Year'].min()}–{cpi['Year'].max()}")
print(
    f"\nInflation descriptive stats "
    f"(all history from {cpi['Year'].min()}):"
)
print(cpi["Inflation"].describe().round(2).to_string())
# Rolling 30-yr average inflation: the COLA draw in
# 03b/04/05 is held for a full career + retirement, so
# it is parameterized from the distribution of long-run
# averages, not annual inflation (see fit_cola_stats)
roll30 = cpi["Inflation"].rolling(30).mean().dropna()
print(
    f"\nRolling 30-yr average inflation (n={len(roll30)}):"
)
print(f"  mean:  {roll30.mean():.2f}%")
print(f"  std:   {roll30.std():.2f}%")
print(
    f"  range: {roll30.min():.2f}%–{roll30.max():.2f}%"
)
print("DoD actuarial long-term assumption: 2.75%")
cpi.tail(10).round(2)

CPI year range: 1913–2025

Inflation descriptive stats (all history from 1913):
count    112.00
mean       3.26
std        4.65
min      -10.50
25%        1.30
50%        2.75
75%        4.46
max       17.97

Rolling 30-yr average inflation (n=83):
  mean:  3.39%
  std:   1.27%
  range: 0.78%–5.44%
DoD actuarial long-term assumption: 2.75%


,Year,Annual,Inflation
103,2016,240.01,1.26
104,2017,245.12,2.13
105,2018,251.11,2.44
106,2019,255.66,1.81
107,2020,258.81,1.23
108,2021,270.97,4.70
109,2022,292.65,8.00
110,2023,304.70,4.12
111,2024,313.69,2.95
112,2025,321.94,2.63


## 6. Life Expectancy (`LifeExpectancy_2022.csv`)

In [9]:
life_exp = pd.read_csv(RAW / "LifeExpectancy_2022.csv")

print(f"Shape: {life_exp.shape}")
print(f"Age range: {life_exp['Age'].min()}–{life_exp['Age'].max()}")

# Spot-checks at representative separation ages
check_ages = [
    (38, "enlisted, 20 YOS"),
    (42, "officer, 20 YOS"),
    (44, "prior-enlisted officer, 20 YOS"),
    (54, "officer, 32 YOS"),
]
print("\nExpected total age at death (conditional on survival to separation):")
hdr = (
    f"  {'Age':<6} {'Profile':<35}"
    f" {'Male total age':>15} {'Female total age':>17}"
)
print(hdr)
print("  " + "-" * 75)
for age, label in check_ages:
    row = life_exp.loc[life_exp["Age"] == age].squeeze()
    male = row['MaleTotalAge']
    female = row['FemaleTotalAge']
    print(f"  {age:<6} {label:<35} {male:>15.1f} {female:>17.1f}")

Shape: (120, 5)
Age range: 0–119

Expected total age at death (conditional on survival to separation):
  Age    Profile                              Male total age  Female total age
  ---------------------------------------------------------------------------
  38     enlisted, 20 YOS                               77.4              81.7
  42     officer, 20 YOS                                77.9              82.0
  44     prior-enlisted officer, 20 YOS                 78.2              82.2
  54     officer, 32 YOS                                79.8              83.2


## 7. Save Processed Files

In [10]:
basic_pay.to_csv(PROCESSED / "basic_pay.csv")
promotion.to_csv(PROCESSED / "promotion_timing.csv", index=False)
withdrawal.to_csv(PROCESSED / "withdrawal_rates.csv", index=False)
tsp.to_csv(PROCESSED / "tsp_returns.csv", index=False)
cpi.to_csv(PROCESSED / "cpi_inflation.csv", index=False)
life_exp.to_csv(PROCESSED / "life_expectancy.csv", index=False)

print("Saved to data/processed/:")
for f in sorted(PROCESSED.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<30} {size_kb:>6.1f} KB")

Saved to data/processed/:
  basic_pay.csv                     3.4 KB
  cpi_inflation.csv                 3.3 KB
  deterministic_results.csv         8.2 KB
  fiscal_results.csv                4.5 KB
  high_three_matrix.csv             0.7 KB
  life_expectancy.csv               3.3 KB
  mc_results.csv                   18.5 KB
  pay_profiles.csv                  2.6 KB
  promotion_timing.csv              0.6 KB
  scenario_weights.csv              1.8 KB
  tsp_returns.csv                   3.3 KB
  withdrawal_rates.csv              1.4 KB
